In [1]:
import pandas as pd
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
from pathlib import Path
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
from sklearn.metrics import brier_score_loss

In [2]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CSV_PATH = "/kaggle/input/datasets/darivi/upside-down-dataset/upside_down_dataset/train.csv"
IMAGE_DIR = "/kaggle/input/datasets/darivi/upside-down-dataset/upside_down_dataset/train"
TEST_DIR = "/kaggle/input/datasets/darivi/upside-down-dataset/upside_down_dataset/test"
OUTPUT_CSV = "submission.csv"
IMG_SIZE = (64, 224)
BATCH_SIZE = 64

print("Device:", DEVICE)

Device: cuda


In [3]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("Device:", device)

Device: cuda


## Dataset

In [4]:
df = pd.read_csv(CSV_PATH)

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["angle"]
)

In [5]:
# train_transform = transforms.Compose([
#     transforms.Resize(IMG_SIZE),
#     transforms.RandomApply([
#         transforms.RandomRotation(degrees=7),
#     ], p=0.5),
#     transforms.ColorJitter(brightness=0.2, contrast=0.2),
#     transforms.ToTensor(),
#     transforms.Normalize(mean=[0.485, 0.456, 0.406],
#                          std=[0.229, 0.224, 0.225]),
#     transforms.RandomErasing(p=0.25),
# ])


train_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.RandomApply([
        transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5))
    ], p=0.3),
    transforms.RandomAffine(
        degrees=5,                    # маленький поворот ±5°
        translate=(0.02, 0.02),       # сдвиг 2%
        scale=(0.95, 1.05)            # масштаб ±5%
    ),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2
    ),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.1))  # стирает кусок картинки
])

val_transform = train_transform
test_transform = train_transform

In [6]:
class TextOrientationDataset(Dataset):

    def __init__(self, df, image_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.image_dir = Path(IMAGE_DIR)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        if row["angle"] == 0:
            image_path = (self.image_dir / "0" / row["filename"])
        else: 
            image_path = (self.image_dir / "180" / row["filename"])

        image = Image.open(image_path).convert("RGB")
        label = 0.0 if row["angle"] == 0 else 1.0

        if self.transform:
            image = self.transform(image)

        return image, label

class TestDataset(Dataset):
    def __init__(self, image_dir, transform=None):
        self.image_dir = Path(image_dir)
        self.files = sorted([f for f in os.listdir(self.image_dir)])
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        fname = self.files[idx]
        img = Image.open(self.image_dir / fname).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, fname

In [7]:
train_dataset = TextOrientationDataset(train_df, IMAGE_DIR, train_transform)
val_dataset = TextOrientationDataset(val_df, IMAGE_DIR, val_transform)
test_dataset = TestDataset(TEST_DIR, test_transform)


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    # persistent_workers=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    # persistent_workers=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    # pin_memory=True
)

loaders = {
    "train": train_loader,
    "valid": val_loader,
    "test": test_loader
}

print("Train:", len(train_dataset))
print("Val:", len(val_dataset))
print("Test:", len(test_dataset))

Train: 16000
Val: 4000
Test: 20000


## Функции для работы с моделью

In [8]:
def brier_score_fn(y_pred, y_true):
    return brier_score_loss(y_true, y_pred)

In [9]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for x_batch, y_batch in loader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device).float()

        optimizer.zero_grad()
        outp = model(x_batch).squeeze(1)
        loss = criterion(outp, y_batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * y_batch.size(0)
        probs = torch.sigmoid(outp)
        preds = (probs > 0.5).long()
        correct += (preds == y_batch.long()).sum().item()
        total += y_batch.size(0)


    return total_loss / total, correct / total

In [10]:
def val_one_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for x_batch, y_batch in loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device).float()

            outp = model(x_batch).squeeze(1)
            loss = criterion(outp, y_batch)
            total_loss += loss.item() * y_batch.size(0)

            probs = torch.sigmoid(outp)
            preds = (probs > 0.5).long()
            correct += (preds == y_batch.long()).sum().item()
            total += y_batch.size(0)

            all_probs.append(probs.cpu().numpy())
            all_labels.append(y_batch.cpu().numpy())

    all_probs = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)
    br_score = brier_score_fn(y_pred=all_probs, y_true=all_labels)

    return total_loss / total, correct / total, br_score

In [11]:
def all_epoch_cycle(model, loaders, optimizer, criterion, max_epochs, device, scheduler,
                    patience=3, min_delta=1e-4):
    model.to(device)

    history = {
        'train_loss': [],
        'train_acc': [],
        'train_br': [],
        'test_loss': [],
        'test_acc': [],
        'test_br': [],
    }

    best_loss = float('inf')
    epochs_no_improve = 0

    for epoch in range(max_epochs):
        train_loss, train_acc = train_one_epoch(model, loaders["train"], optimizer, criterion, device)
        test_loss, test_acc, test_br = val_one_epoch(model, loaders["valid"], criterion, device)

        scheduler.step()

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['test_loss'].append(test_loss)
        history['test_acc'].append(test_acc)
        history['test_br'].append(test_br)

        print(f"Epoch {epoch+1}: Train Loss = {train_loss:.4f}, Test Loss = {test_loss:.4f}")
        print(f"         Train Acc = {train_acc:.4f}, Test Acc = {test_acc:.4f}    Test Brier = {test_br:.4f} \n")

        
        if test_loss < best_loss - min_delta:
            best_loss = test_loss
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping on epoch {epoch+1} (best test_loss = {best_loss:.4f})")
                break

    return history, epoch + 1

In [12]:
def plot_metrics(history, max_epochs):
    epochs = range(1, max_epochs + 1)

    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(epochs, history['train_loss'], 'b-', label='Train Loss')
    plt.plot(epochs, history['test_loss'], 'r-', label='Test Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training and Validation Loss')
    plt.legend()
    plt.grid(True)

    plt.subplot(1, 2, 2)
    plt.plot(epochs, history['train_acc'], 'b-', label='Train Accuracy')
    plt.plot(epochs, history['test_acc'], 'r-', label='Test Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.title('Training and Validation Accuracy')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()

# Модель

In [13]:
num_epochs = 30
lr_head = 1e-3
lr_backbone = 1e-4
weight_decay = 1e-4

In [14]:
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, 1)

# model = models.efficientnet_b3(weights="IMAGENET1K_V1")
# model.classifier[1] = nn.Linear(model.classifier[1].in_features, 1)
model = model.to(device)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 121MB/s] 


In [15]:
backbone_params = []
head_params = []
for name, p in model.named_parameters():
    if name.startswith("fc."):
        head_params.append(p)
    else:
        backbone_params.append(p)

optimizer = torch.optim.AdamW([
    {"params": backbone_params, "lr": 1e-5},
    {"params": head_params, "lr": 1e-3}
], weight_decay=1e-4)

criterion = nn.BCEWithLogitsLoss()
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

In [16]:
history, max_epochs = all_epoch_cycle(
    model, 
    loaders,
    optimizer, 
    criterion, 
    max_epochs=num_epochs,
    device=device,
    scheduler=scheduler,
)

Epoch 1: Train Loss = 0.6964, Test Loss = 0.6827
         Train Acc = 0.5391, Test Acc = 0.5533    Test Brier = 0.2449 

Epoch 2: Train Loss = 0.6511, Test Loss = 0.6166
         Train Acc = 0.5859, Test Acc = 0.6205    Test Brier = 0.2168 

Epoch 3: Train Loss = 0.5951, Test Loss = 0.5896
         Train Acc = 0.6418, Test Acc = 0.6512    Test Brier = 0.2059 

Epoch 4: Train Loss = 0.5565, Test Loss = 0.5409
         Train Acc = 0.6821, Test Acc = 0.6943    Test Brier = 0.1861 

Epoch 5: Train Loss = 0.5275, Test Loss = 0.5291
         Train Acc = 0.7123, Test Acc = 0.7043    Test Brier = 0.1804 

Epoch 6: Train Loss = 0.4982, Test Loss = 0.4980
         Train Acc = 0.7296, Test Acc = 0.7278    Test Brier = 0.1691 

Epoch 7: Train Loss = 0.4774, Test Loss = 0.4828
         Train Acc = 0.7459, Test Acc = 0.7432    Test Brier = 0.1626 

Epoch 8: Train Loss = 0.4525, Test Loss = 0.4555
         Train Acc = 0.7631, Test Acc = 0.7635    Test Brier = 0.1513 

Epoch 9: Train Loss = 0.4335, Te

# Submission

In [20]:
model.eval()
model.to(device)

all_ids = []
all_probs = []

with torch.no_grad():
    for x_batch, filenames in tqdm(test_loader, desc="Predicting"):
        x_batch = x_batch.to(device)
        logits = model(x_batch).squeeze(1)
        probs = torch.sigmoid(logits)

        all_probs.extend(probs.cpu().numpy().tolist())
        all_ids.extend(f[:-4] for f in filenames)


Predicting:   0%|          | 0/313 [00:00<?, ?it/s]

In [21]:
submission = pd.DataFrame({
    "image_id": all_ids,
    "p_180": all_probs
})

submission.to_csv(OUTPUT_CSV, index=False)
print(f"\nСохранено: {OUTPUT_CSV}")
print(submission.head())
print(f"\nРазмер: {submission.shape}")
print(f"Средняя вероятность: {submission['p_180'].mean():.4f}")
print(f"Мин/Макс: {submission['p_180'].min():.4f} / {submission['p_180'].max():.4f}")


Сохранено: submission.csv
     image_id     p_180
0  test_00000  0.878749
1  test_00001  0.942604
2  test_00002  0.528629
3  test_00003  0.722732
4  test_00004  0.979130

Размер: (20000, 2)
Средняя вероятность: 0.6800
Мин/Макс: 0.0000 / 1.0000
